In [ ]:
import sys, os, shutil, tempfile, warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._forking_sequences import fork_sequences
from dataloaders._ts_dataloader      import DataLoaderFactory, FullSeriesDataset
from dataloaders._ts_sharding        import (
    write_sharded_dataset,
    ShardedTrainDataset,
    ShardedValDataset,
    ShardedTestDataset,
)
from common._base_model import BaseModel
from common.train import train, train_distributed, eval_test
from models.linear import TinyLinearModel

warnings.filterwarnings("ignore")
print("imports OK")

In [ ]:
# ── Synthetic dataframe factory ───────────────────────────────────────────────

def make_df(
    n_series: int = 3,
    n_steps:  int = 500,
    n_hist:   int = 1,       # number of hist exog cols
    seed:     int = 42,
) -> pd.DataFrame:
    """
    Returns a long-format DataFrame with columns:
        unique_id, ds, y, available_mask, hist_0, hist_1, ...
    All series have equal length (n_steps timesteps).
    available_mask is 1 everywhere (no gaps).
    """
    rng   = np.random.default_rng(seed)
    times = pd.date_range("2020-01-01", periods=n_steps, freq="5min")
    rows  = []
    for s in range(n_series):
        y    = rng.standard_normal(n_steps).astype(np.float32)
        hist = {f"hist_{i}": rng.standard_normal(n_steps).astype(np.float32)
                for i in range(n_hist)}
        df_s = pd.DataFrame({"unique_id": f"series_{s}", "ds": times, "y": y,
                              "available_mask": np.ones(n_steps, dtype=np.float32),
                              **hist})
        rows.append(df_s)
    return pd.concat(rows, ignore_index=True)


# ── Config factories ──────────────────────────────────────────────────────────

def make_mcfg(
    context_length:    int  = 64,
    fcd_samples:       int  = 4,
    batch_size:        int  = 2,
    max_steps:         int  = 20,
    val_check_interval:int  = 10,
    mixing_strategy:   str  = "concat",
    normalize:         bool = False,
    checkpoint_dir:    str  = "/tmp/ckpts",
    num_workers:       int  = 0,
):
    return SimpleNamespace(
        context_length         = context_length,
        input_size             = context_length,
        fcd_samples            = fcd_samples,
        batch_size             = batch_size,
        valid_batch_size       = batch_size,
        max_steps              = max_steps,
        val_check_interval     = val_check_interval,
        learning_rate          = 1e-3,
        gradient_clip_val      = 1.0,
        early_stopping_patience= 9999,  # disable for tests
        monitor_metric         = "loss",
        monitor_mode           = "min",
        mixing_strategy        = mixing_strategy,
        drop_last              = False,
        normalize              = normalize,
        batch_mode             = "full_series",
        checkpoint_dir         = checkpoint_dir,
        checkpoint_step        = 99999,  # suppress mid-run saves
        num_workers            = num_workers,
    )


def make_entry(
    path:            str,
    name:            str  = "ds",
    horizon:         int  = 6,
    val_size:        int  = 50,
    test_size:       int  = 50,
    weight:          float= 1.0,
    hist_exog_cols:  list = None,
    futr_exog_cols:  list = None,
    stat_exog_cols:  list = None,
    per_series_split:bool = False,
    use_context_head:bool = True,
    sharded_dir:     str  = None,
):
    return SimpleNamespace(
        path             = path,
        name             = name,
        horizon          = horizon,
        val_size         = val_size,
        test_size        = test_size,
        weight           = weight,
        hist_exog_cols   = hist_exog_cols  or [],
        futr_exog_cols   = futr_exog_cols  or [],
        stat_exog_cols   = stat_exog_cols  or [],
        per_series_split = per_series_split,
        use_context_head = use_context_head,
        sharded_dir      = sharded_dir,
    )


def make_dcfg(train_entries, val_entries=None, test_entries=None):
    return SimpleNamespace(
        train      = train_entries,
        validation = val_entries  or train_entries,
        test       = test_entries or train_entries,
    )


print("helpers OK")

In [ ]:
def make_model(mcfg, n_channels=3, n_hist=0):
    return TinyLinearModel(
        context_length = mcfg.context_length,
        horizon        = 6,
        n_channels     = n_channels,
        n_hist         = n_hist,
    )

## Test 8 — `fork_sequences` unit tests

In [ ]:
print("=" * 60)
print("TEST 8 — fork_sequences shape assertions")
print("=" * 60)

def make_batch(B=2, T=200, C=3, Vh=1, H=6, device="cpu"):
    """Minimal collated batch as _full_series_collate_fn would produce."""
    return {
        "x_enc":          torch.randn(B, T, C, 1 + Vh, device=device),
        "available_mask": torch.ones(B, C, T, device=device),
        "channel_mask":   torch.ones(B, C, device=device),
        "horizon":        torch.full((B,), H, dtype=torch.long, device=device),
    }

L, H, S = 32, 6, 1

# ── Training path (fcd_samples=4) ─────────────────────────────────────────────
print("\n--- Training path (fcd_samples=4) ---")
batch = make_batch(B=2, T=200, C=3, Vh=1, H=H)
out   = fork_sequences(batch, context_length=L, fcd_samples=4, horizon=H)

B_out, enc_size, C_out, feat = out["insample_y"].shape
print(f"insample_y:    {tuple(out['insample_y'].shape)}    expected [2, enc_size, 3, 2]")
print(f"outsample_y:   {tuple(out['outsample_y'].shape)}   expected [2, 4, {H}, 3]")
print(f"available_mask:{tuple(out['available_mask'].shape)}")

assert out["outsample_y"].shape == (2, 4, H, 3), \
    f"outsample_y shape wrong: {out['outsample_y'].shape}"
assert out["available_mask"].shape == (2, 3, enc_size), \
    f"available_mask shape wrong: {out['available_mask'].shape}"
print("training path shapes ✓")

# ── Val/test path (fcd_samples=-1) ────────────────────────────────────────────
print("\n--- Val/test path (fcd_samples=-1) ---")
T = 200
batch_val  = make_batch(B=2, T=T, C=3, Vh=0, H=H)
out_val    = fork_sequences(batch_val, context_length=L, fcd_samples=-1, horizon=H)
n_expected = T-L-H+1

print(f"n_valid_fcds({T}, L={L}, H={H}, S={S}) = {n_expected}")
print(f"outsample_y:   {tuple(out_val['outsample_y'].shape)}  expected [2, {n_expected}, {H}, 3]")
assert out_val["outsample_y"].shape[1] == n_expected, \
    f"n_fcds wrong: {out_val['outsample_y'].shape[1]} != {n_expected}"
print("val/test path shapes ✓")

# ── Left-padding: windows never start in padding ───────────────────────────────
print("\n--- Left-padding: sampler skips padded timesteps ---")
T_real, T_pad = 100, 50
T_total       = T_real + T_pad
mask = torch.zeros(1, 3, T_total)
mask[:, :, T_pad:] = 1.0   # real data starts at T_pad

batch_pad = {
    "x_enc":          torch.randn(1, T_total, 3, 1),
    "available_mask": mask,
    "horizon":        torch.tensor([H]),
}
from dataloaders._forking_sequences import heterogeneous_sampler
for _ in range(20):
    ws, _ = heterogeneous_sampler(mask, L, fcd_samples=4, horizon=H)
    assert ws[0].item() >= T_pad, \
        f"window_start {ws[0].item()} < T_pad {T_pad} — sampled in padding!"
print("all 20 sampled window_starts ≥ T_pad ✓")

# ── available_mask shape assertion ────────────────────────────────────────────
print("\n--- available_mask shape mismatch raises AssertionError ---")
bad_mask_batch = {
    "x_enc":          torch.randn(2, 100, 3, 1),
    "available_mask": torch.ones(2, 100),   # wrong: [B, T] instead of [B, C, T]
    "horizon":        torch.tensor([H, H]),
}
try:
    fork_sequences(bad_mask_batch, context_length=L, fcd_samples=4, horizon=H)
    assert False, "should have raised"
except AssertionError as e:
    print(f"raised AssertionError as expected: {e}")

print("\n✓ TEST 8 PASSED")